# Metric Evaluation: WNBA Salary Valuation

## Notebook Outline

1. Load merged salary, player statistics, and team data
2. Reuse baseline models from the `assessing_learnability_multiyear` notebook
3. Evaluate models using RMSE, MAPE, and CPWS
4. Compare Dummy, Linear Regression, and Random Forest performance
5. Visualize cross-validated metric results
6. Identify the primary metric for player salary prediction
7. Summarize baseline evaluation findings and limitations

In [2]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

Note: `cross_validate` is added to support evaluating multiple metrics (RMSE, MAE, and R²) in the same cross-validation loop.

## 1. Load Data

We use the same merged multi-year dataset from the `assessing_learnability_multiyear` notebook.

In [3]:
# Load and integrate data
repo = Path.cwd().parent
data_dir = repo / "data" / "raw"
years = list(range(2021, 2026))

# Load raw CSV components
raw = {}
for year in years:
    raw[year] = {
        "advanced": pd.read_csv(data_dir / f"{year}_advanced.csv"),
        "per_game": pd.read_csv(data_dir / f"{year}_per_game.csv"),
        "totals": pd.read_csv(data_dir / f"{year}_totals.csv"),
        "salary": pd.read_csv(data_dir / f"salary_{year}.csv"),
        "team_advanced": pd.read_csv(data_dir / f"{year}_advanced-team.csv"),
        "standings": pd.read_csv(data_dir / f"{year}_wnba_standings.csv"),
    }

for year, files in raw.items():
    print(f"\n{year}")
    for name, df in files.items():
        print(f"{name}: {df.shape}")


2021
advanced: (155, 24)
per_game: (155, 27)
totals: (155, 26)
salary: (191, 27)
team_advanced: (12, 27)
standings: (12, 5)

2022
advanced: (166, 24)
per_game: (166, 27)
totals: (166, 26)
salary: (202, 27)
team_advanced: (12, 27)
standings: (12, 5)

2023
advanced: (156, 24)
per_game: (156, 27)
totals: (156, 26)
salary: (184, 27)
team_advanced: (12, 27)
standings: (12, 5)

2024
advanced: (157, 24)
per_game: (157, 27)
totals: (157, 26)
salary: (180, 27)
team_advanced: (12, 27)
standings: (12, 5)

2025
advanced: (182, 24)
per_game: (182, 27)
totals: (182, 26)
salary: (227, 27)
team_advanced: (13, 27)
standings: (13, 5)


## Basic Cleaning

Before merging the datasets, we standardize column names and clean simple text values.

In [4]:
# Standardize column names into a code-friendly format
def clean_col(name):
    text = str(name).strip().lower()
    text = re.sub(r"20\d{2}\s+salary", "salary", text)
    text = re.sub(r"20\d{2}\s+signing", "signing", text)
    text = text.replace("%", "pct")

    # Replace spaces and symbols with underscores
    text = re.sub(r"[^0-9a-z]+", "_", text)
    text = text.strip("_")
    return text

# Apply basic cleaning to one dataframe
def clean_df(df):
    df = df.copy()

    # Apply the same column-name format to each dataset
    df.columns = [clean_col(col) for col in df.columns]

    # Clean text columns by removing extra spaces and standardizing missing values
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"": np.nan, "—": np.nan, "nan": np.nan})

    return df

cleaned = {}
for year, files in raw.items():
    cleaned[year] = {name: clean_df(df) for name, df in files.items()}
    cleaned[year]["salary"]["salary"] = pd.to_numeric(
        cleaned[year]["salary"]["salary"],
        errors="coerce"
    )

## Final Data Merge and Name Matching

Now, we create the final modeling dataframe by merging player salary, player production, and team-level context for each season.

This step follows the same merge and name-matching logic from the `assessing_learnability_multiyear` notebook, so the metric evaluation uses the same baseline modeling dataset.

In [5]:
# Map full team names from team-level files to the abbreviations used in player-level files
teammap = {
    "Atlanta Dream": "ATL",
    "Chicago Sky": "CHI",
    "Connecticut Sun": "CON",
    "Dallas Wings": "DAL",
    "Golden State Valkyries": "GSV",
    "Indiana Fever": "IND",
    "Las Vegas Aces": "LVA",
    "Los Angeles Sparks": "LAS",
    "Minnesota Lynx": "MIN",
    "New York Liberty": "NYL",
    "Phoenix Mercury": "PHO",
    "Seattle Storm": "SEA",
    "Washington Mystics": "WAS",
}

def team_clean(val):
    if pd.isna(val):
        return np.nan

    # Basketball Reference marks some teams with "*"; remove it before mapping
    return str(val).replace("*", "").strip()

season_parts = {}

for year in years:
    adv = cleaned[year]["advanced"]
    per = cleaned[year]["per_game"]
    tot = cleaned[year]["totals"]
    sal = cleaned[year]["salary"].copy()
    teamadv = cleaned[year]["team_advanced"].copy()
    stand = cleaned[year]["standings"].copy()

    # Standardize team names so team-level data can merge with player-level data
    teamadv["team_name"] = teamadv["team"].map(team_clean)
    teamadv["team"] = teamadv["team_name"].map(teammap)
    stand["team_name"] = stand["team_name"].map(team_clean)
    stand["team"] = stand["team_name"].map(teammap)

    # Combine team advanced stats with standings data
    teamdf = teamadv.merge(stand, on=["team", "team_name"], how="left", suffixes=("_adv", "_stand"))

    # Merge per_game and advanced player stats into one production dataframe
    playerdf = per.merge(adv, on=["player", "team", "pos", "g", "mp"], how="outer", suffixes=("_per", "_adv"))
    # Add total count stats to the same player production dataframe.
    playerdf = playerdf.merge(tot, on=["player", "team", "pos", "g", "mp", "gs"], how="outer", suffixes=("", "_tot"))

    # Check player name mismatches before creating final_df
    salary_only = sal[~sal["player"].isin(playerdf["player"])]
    stats_only = playerdf[~playerdf["player"].isin(sal["player"])]

    print(f"\n{year} Before name fixes")
    print("Salary only players:")
    print(sorted(salary_only["player"].dropna().unique()))

    print("\nStats only players:")
    print(sorted(stats_only["player"].dropna().unique()))

    # Save the season-level intermediate dataframes so they can be reused
    season_parts[year] = {
        "sal": sal,
        "playerdf": playerdf,
        "teamdf": teamdf,
    }


2021 Before name fixes
Salary only players:
['Brittany Boyd-Jones', 'Jocelyn Willoughby', 'Maria Kliundikova', "N'Dea Jones", 'Rennia Davis']

Stats only players:
['Brittany Boyd']

2022 Before name fixes
Salary only players:
['Asia (AD) Durr', 'Brittney Griner', 'Kia Nurse']

Stats only players:
['AD Durr']

2023 Before name fixes
Salary only players:
['Asia (AD) Durr', 'Azurá Stevens', 'Christyn Williams', 'Diamond DeShields', 'Dorka Juhász', 'Isabelle Harrison', 'Ivana Dojkić', 'Katie Lou Samuelson', 'Lou Lopez Sénéchal', 'Natalie Achonwa', 'Riquna Williams', 'Sika Koné', 'Skylar Diggins']

Stats only players:
['AD Durr', 'Azura Stevens', 'Dorka JuhÃ¡sz', 'Ivana DojkiÄ\x87', 'Kadi Sissoko', 'Sika KonÃ©']

2024 Before name fixes
Salary only players:
['Dorka Juhász', 'Ivana Dojkić', 'Lou Lopez Sénéchal', 'Nika Mühl', 'Olivia Époupa', 'Sika Kone', 'Temi Fágbénlé']

Stats only players:
['Dorka JuhÃ¡sz', 'Ivana DojkiÄ\x87', 'Lou Lopez SÃ©nÃ©chal', 'Nika MÃ¼hl', 'Olivia Ã\x89poupa', 'Sik

In [6]:
# Manually fix clear name encoding/spelling mismatches across the 2021-2025 files
name_fix = {
    "AD Durr": "Asia (AD) Durr",
    "Anastasiia Kosu": "Anastasiia Olairi Kosu",
    "Azura Stevens": "Azura Stevens",
    "Azurá Stevens": "Azura Stevens",
    "Brittany Boyd": "Brittany Boyd-Jones",
    "Dorka JuhÃ¡sz": "Dorka Juhasz",
    "Dorka Juhász": "Dorka Juhasz",
    "Ivana DojkiÄ": "Ivana Dojkic",
    "Ivana Dojkić": "Ivana Dojkic",
    "Janelle SalaÃ¼n": "Janelle Salaun",
    "Janelle Salaün": "Janelle Salaun",
    "LeÃ¯la Lacan": "Leila Lacan",
    "Leïla Lacan": "Leila Lacan",
    "Lou Lopez SÃ©nÃ©chal": "Lou Lopez Senechal",
    "Lou Lopez Sénéchal": "Lou Lopez Senechal",
    "Luisa GeiselsÃ¶der": "Luisa Geiselsoder",
    "Luisa Geiselsöder": "Luisa Geiselsoder",
    "Mamignan TourÃ©": "Mamignan Toure",
    "Mamignan Touré": "Mamignan Toure",
    "MariÃ¨me Badiane": "Marieme Badiane",
    "Marième Badiane": "Marieme Badiane",
    "Nika MÃ¼hl": "Nika Muhl",
    "Nika Mühl": "Nika Muhl",
    "Olivia Ãpoupa": "Olivia Epoupa",
    "Olivia Époupa": "Olivia Epoupa",
    "Sika KonÃ©": "Sika Kone",
    "Sika Koné": "Sika Kone",
    "Te-Hina PaoPao": "Te-Hina Paopao",
    "Temi Fágbénlé": "Temi Fagbenle",
}

season_dfs = []

for year in years:
    # Retrieve the saved season-level dataframes for applying name fixes and final merging
    sal = season_parts[year]["sal"].copy()
    playerdf = season_parts[year]["playerdf"].copy()
    teamdf = season_parts[year]["teamdf"]

    # Apply the name corrections before merging salary and production data
    sal["player"] = sal["player"].replace(name_fix)
    playerdf["player"] = playerdf["player"].replace(name_fix)

    # Re-check after name fixes
    salary_only = sal[~sal["player"].isin(playerdf["player"])]
    stats_only = playerdf[~playerdf["player"].isin(sal["player"])]

    print(f"\n{year} After name fixes")
    print("Salary only players:")
    print(sorted(salary_only["player"].dropna().unique()))

    print("\nStats only players:")
    print(sorted(stats_only["player"].dropna().unique()))

    # seasondf = salary + player stats + team context + season column
    seasondf = sal.merge(playerdf, on="player", how="inner", suffixes=("_sal", ""))
    seasondf = seasondf.merge(teamdf, on="team", how="left", suffixes=("", "_team"))
    seasondf["season"] = year

    # season_dfs = [seasondf_2021, seasondf_2022, seasondf_2023, seasondf_2024, seasondf_2025]
    season_dfs.append(seasondf)

final_df = pd.concat(season_dfs, ignore_index=True)

print(final_df.shape)
final_df.head()


2021 After name fixes
Salary only players:
['Jocelyn Willoughby', 'Maria Kliundikova', "N'Dea Jones", 'Rennia Davis']

Stats only players:
[]

2022 After name fixes
Salary only players:
['Brittney Griner', 'Kia Nurse']

Stats only players:
[]

2023 After name fixes
Salary only players:
['Christyn Williams', 'Diamond DeShields', 'Isabelle Harrison', 'Katie Lou Samuelson', 'Lou Lopez Senechal', 'Natalie Achonwa', 'Riquna Williams', 'Skylar Diggins']

Stats only players:
['Kadi Sissoko']

2024 After name fixes
Salary only players:
[]

Stats only players:
[]

2025 After name fixes
Salary only players:
['Georgia Amoore', 'Katie Lou Samuelson']

Stats only players:
[]
(967, 124)


,player,salary,signing,g_sal,gs_sal,min,pts_sal,fgm,fga_sal,fgpct,...,opp_tov_pct,drb_pct,opp_ft_rate,arena_name,team_name,wins_stand,losses_stand,win_loss_pct,gb,season
0,Tina Charles,175000,UFA,27,27,33.3,23.4,8.8,19.6,44.9%,...,15.3,76.4,0.208,St. Elizabeths East Arena,Washington Mystics,12.0,20.0,0.375,14.0,2021
1,Brittney Griner,221450,NaN,30,30,32.8,20.5,8.3,14.4,57.5%,...,11.3,75.1,0.167,Phoenix Suns Arena,Phoenix Mercury,19.0,13.0,0.594,7.0,2021
2,Breanna Stewart,190550,NaN,28,28,33.3,20.3,6.9,15.8,43.9%,...,14.4,78.6,0.180,Angels of the Winds Arena,Seattle Storm,21.0,11.0,0.656,5.0,2021
3,Jonquel Jones,190550,NaN,27,27,31.7,19.4,7.1,13.8,51.5%,...,16.1,82.1,0.201,Mohegan Sun Arena,Connecticut Sun,26.0,6.0,0.813,NaN,2021
4,Arike Ogunbowale,58710,NaN,32,32,31.4,18.7,6.2,16.3,38.3%,...,14.1,77.6,0.206,College Park Center,Dallas Wings,14.0,18.0,0.438,12.0,2021


## 2. Metric Evaluation Setup

This section reuses the baseline modeling setup from the `assessing_learnability_multiyear` notebook.

The goal is not to build a new model. Instead, we evaluate the same baseline models using metrics aligned with the project KPI file.

Setup:

- `x`: model input features
- `y`: target variable, salary
- Models: `DummyRegressor`, `LinearRegression`, and `RandomForestRegressor`
- Validation method: 5-fold cross-validation

The `DummyRegressor` is the trivial baseline model. It predicts salary using a simple rule, such as the training-set mean salary, without learning relationships between features and salary.

The primary predictive metric is RMSE, which is the project’s primary predictive KPI. RMSE penalizes large salary prediction errors more heavily, which is useful because large salary mispricing errors are costly.

MAPE is used as the secondary fairness-oriented KPI because it measures error relative to contract size. This helps compare model error across lower-salary and higher-salary players.

CPWS is included as the business ROI KPI. Unlike RMSE and MAPE, CPWS is not a direct cross-validation scoring metric for individual salary prediction. Instead, it measures team-level spending efficiency as total team payroll divided by total season win shares.